# Exoplanet Discovery Methods: A Data-Driven Analysis

## Patterns and Selection Effects in NASA's Exoplanet Archive

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

In [ ]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = 'data/exoplanet_dataset/data/exoplanets_pscomppars_2026-09-12.csv'
RESULTS_DIR = 'results/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Data: {DATA_PATH}')
print(f'Output: {RESULTS_DIR}')

In [ ]:
def load_and_validate_data(path, target_methods):
    raw_df = pd.read_csv(path)
    filtered_df = raw_df[raw_df['discoverymethod'].isin(target_methods)].copy()
    return raw_df, filtered_df

TARGET_METHODS = ['Transit', 'Radial Velocity', 'Imaging', 'Microlensing']
raw_df, project_df = load_and_validate_data(DATA_PATH, TARGET_METHODS)
print(f'Loaded {len(project_df)} planets')
print(project_df['discoverymethod'].value_counts())

In [ ]:
df_prepared = project_df.copy()
log_features = {'pl_orbper': 'log10_pl_orbper', 'pl_rade': 'log10_pl_rade',
                'pl_bmasse': 'log10_pl_bmasse', 'pl_eqt': 'log10_pl_eqt', 'sy_dist': 'log10_sy_dist'}
for orig, log in log_features.items():
    df_prepared[log] = np.log10(df_prepared[orig])
print('Log features created')

In [ ]:
measurement_cols = ['pl_orbper', 'pl_rade', 'pl_bmasse', 'pl_eqt', 'st_teff', 'st_rad', 'sy_dist']
for col in measurement_cols:
    df_prepared[f'has_{col}'] = (~df_prepared[col].isna()).astype(int)
print('Missingness indicators created')

In [ ]:
np.random.seed(42)
groups = df_prepared['hostname'].unique()
np.random.shuffle(groups)
n_groups = len(groups)
train_cut, val_cut = int(0.6 * n_groups), int(0.8 * n_groups)

train_df = df_prepared[df_prepared['hostname'].isin(groups[:train_cut])].copy()
val_df = df_prepared[df_prepared['hostname'].isin(groups[train_cut:val_cut])].copy()
test_df = df_prepared[df_prepared['hostname'].isin(groups[val_cut:])].copy()

y_train, y_val, y_test = train_df['discoverymethod'], val_df['discoverymethod'], test_df['discoverymethod']
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_prepared['discoverymethod'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Exoplanet Count by Discovery Method')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}01_class_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
properties = [('log10_pl_orbper', 'Orbital Period'), ('log10_pl_rade', 'Planet Radius'), ('log10_pl_bmasse', 'Planet Mass')]
for idx, (feature, label) in enumerate(properties):
    data = [df_prepared[df_prepared['discoverymethod'] == m][feature].dropna() for m in TARGET_METHODS]
    axes[idx].boxplot(data)
    axes[idx].set_xticklabels(TARGET_METHODS)
    axes[idx].set_title(label)
    axes[idx].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}02_properties.png', dpi=150)
plt.show()

In [ ]:
feature_sets = {
    'A': ['log10_pl_orbper', 'log10_pl_rade', 'log10_pl_bmasse', 'log10_pl_eqt'],
    'B': ['st_teff', 'st_rad', 'log10_sy_dist'],
    'C': ['log10_pl_orbper', 'log10_pl_rade', 'log10_pl_bmasse', 'log10_pl_eqt', 'st_teff', 'st_rad', 'log10_sy_dist']
}
print('Feature sets defined')

In [ ]:
train_val_combined = pd.concat([train_df, val_df])
y_train_val = pd.concat([y_train, y_val])

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(train_df[['pl_orbper']], y_train)
baseline_pred = baseline.predict(val_df[['pl_orbper']])
baseline_f1 = f1_score(y_val, baseline_pred, average='macro', zero_division=0)
print(f'Baseline Macro F1: {baseline_f1:.4f}')

In [ ]:
results = []
for set_id, features in feature_sets.items():
    pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    X_train_val = pipeline.fit_transform(train_val_combined[features])
    X_test = pipeline.transform(test_df[features])
    
    for model_name, model_class in [('KNN', KNeighborsClassifier(n_neighbors=5)), ('Decision Tree', DecisionTreeClassifier(max_depth=10, random_state=42)), ('Random Forest', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42))]:
        model = model_class
        model.fit(X_train_val, y_train_val)
        y_pred = model.predict(X_test)
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)
        results.append({'feature_set': set_id, 'model': model_name, 'macro_f1': macro_f1, 'balanced_accuracy': bal_acc, 'accuracy': acc})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['feature_set', 'macro_f1'], ascending=[True, False])
print(results_df)
results_df.to_csv(f'{RESULTS_DIR}model_results.csv', index=False)

In [ ]:
print('\nAnalysis complete!')
print(f'Results saved to {RESULTS_DIR}')